# 🖼️ 01 - Matplotlib, the Object-Oriented Way

Module 07 ended with four datasets that had **identical** means, standard deviations, and correlations. The numbers said they were the same. Let's look at them:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, StrMethodFormatter
from pathlib import Path

OUT = Path('01-OutputFiles')
OUT.mkdir(exist_ok=True)

anscombe = pd.read_csv('anscombe.csv')

fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=True, sharey=True, layout='constrained')

for ax, (name, data) in zip(axes.flat, anscombe.groupby('dataset')):
    r = data['x'].corr(data['y'])
    ax.scatter(data['x'], data['y'], color='tab:blue', edgecolor='black')
    ax.set_title(f'Dataset {name}   (r = {r:.2f})')

fig.suptitle("Anscombe's Quartet: Same Statistics, Different Data", fontsize=14)
fig.savefig(OUT / 'anscombe_quartet.png', dpi=150, bbox_inches='tight')
plt.show()

Four completely different stories:
- **I**: a noisy straight line. The one case where r = 0.82 tells the truth.
- **II**: a perfect **curve**. A straight-line correlation is the wrong tool.
- **III**: a perfect line, plus **one outlier** dragging r down.
- **IV**: no relationship at all. **One point** creates the entire correlation.

That's why this module exists: **plot first, then summarize.**

That cell also previewed this notebook. It used `fig` and `ax` objects, a loop over subplots, shared axes, and `fig.savefig()`. Foundations 07 taught charts with `plt.` calls. This notebook switches to the object-oriented style, on real DataFrames.

## ✅ Learning Goals
- Explain the difference between the `plt.` (pyplot) style and the object-oriented `fig, ax` style, and why the second one scales
- Plot DataFrame columns onto an `ax`, and set titles and labels with `ax.set()`
- Mark statistics from Module 07 (mean, median, IQR fences) with reference lines
- Use a **log scale** for skewed data, and format ticks as money or percentages
- Annotate a specific point, label bars with their values, and declutter spines and grids
- Build multi-panel figures with a loop, and save them with `fig.savefig()`

## 1. Two Styles, One Library
| | pyplot style (Foundations 07) | object-oriented style (this module) |
|---|---|---|
| Create | `plt.plot(...)` | `fig, ax = plt.subplots()` then `ax.plot(...)` |
| Label | `plt.title()`, `plt.xlabel()` | `ax.set_title()`, `ax.set_xlabel()`, or `ax.set(...)` |
| Which chart does a call affect? | whichever one is "current" (hidden state) | the exact `ax` you call it on |
| Multiple charts | easy to get wrong | natural: one `ax` per chart |
| Save | `plt.savefig()` | `fig.savefig()` |

pyplot keeps a hidden "current axes" and sends every call there. That's fine for one quick chart. With several charts, or a function that draws on a chart someone passes in, explicit objects are clearer: you always know **which** chart you're changing.

The rule for this module:
```text
plt   → create the figure (plt.subplots) and show it (plt.show)
fig   → the whole canvas: suptitle, layout, saving
ax    → one chart: data, titles, labels, limits, ticks
```

## 2. Plotting DataFrame Columns
`ax` methods take any column directly. `ax.set()` sets several properties at once:

In [ ]:
titanic = pd.read_csv('titanic.csv')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(titanic['Age'].dropna(), bins=20, color='tab:blue', edgecolor='white')
ax.set(title='Ages of Titanic Passengers', xlabel='Age (years)', ylabel='Passengers')
plt.show()

`.dropna()` matters: `ax.hist()` can't place a `NaN` in any bin. (Module 07 notebook 02: always know how many values a statistic, or a chart, is based on.)

## 3. Reference Lines for Statistics
`ax.axvline()` and `ax.axhline()` draw a line across the whole chart. They're a natural way to put a Module 07 statistic **on** the chart:

In [ ]:
fare = titanic['Fare']

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(fare, bins=40, color='lightgray', edgecolor='white')
ax.axvline(fare.mean(), color='tab:red', linestyle='--', label=f'mean = {fare.mean():.1f}')
ax.axvline(fare.median(), color='tab:blue', linewidth=2, label=f'median = {fare.median():.1f}')
ax.set(title='Titanic Fares: the Mean Is Pulled Right', xlabel='Fare (£)', ylabel='Passengers')
ax.legend()
plt.show()

This is Module 07's "mean > median means right-skewed," made visible. Almost everyone is piled up on the left, and a long, thin tail stretches past £500.

✅ **Your Turn:** Copy `iqr_fences()` from Module 07 notebook 03 and shade the outlier region of the fare chart with `ax.axvspan(high_fence, fare.max(), color='tab:orange', alpha=0.2)`.

In [ ]:
# Your turn here

## 4. Log Scales for Skewed Data
In the chart above, the tail is so long that the crowded part is squeezed into a few bars. Module 07 notebook 03 tamed skew with a log transform. On a chart, the equivalent is a **log-scale axis**: each step on the axis is a **multiple** (×10) rather than a fixed amount.

For a histogram, use bins that are evenly spaced on the log scale (`np.geomspace`). Log scales can't show zero, so drop the £0 fares and say so:

In [ ]:
paid = fare[fare > 0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(paid, bins=np.geomspace(paid.min(), paid.max(), 30), color='tab:blue', edgecolor='white')
ax.set_xscale('log')
ax.xaxis.set_major_formatter(StrMethodFormatter('£{x:,.0f}'))
ax.set(title=f'Fares on a Log Scale (excludes {(fare == 0).sum()} £0 fares)',
       xlabel='Fare (log scale)', ylabel='Passengers')
plt.show()

Now you can see the structure the first chart hid: clusters of fares around £8, £15–30, and £50–90, which line up with the three classes.

`StrMethodFormatter('£{x:,.0f}')` uses the same format-spec mini-language as f-strings (Module 02): `,` for thousands separators and `.0f` for no decimals.

## 5. Percent Ticks and Bar Labels
For rates (Module 07 notebook 04), format the axis as percentages with `PercentFormatter(xmax=1)` (which means "1.0 is 100%"). `ax.bar_label()` writes each bar's value on the bar:

In [ ]:
rates = titanic.groupby('Pclass')['Survived'].mean()
labels = ['1st', '2nd', '3rd']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, rates.values, color='tab:blue')
ax.bar_label(bars, labels=[f'{r:.0%}' for r in rates], padding=3)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set(title='Survival Rate by Class', xlabel='Ticket class', ylabel='Survived', ylim=(0, 1))
plt.show()

`ylim=(0, 1)` keeps the full 0–100% scale. Bars encode values by **length**, so a bar axis should start at zero (more on honest scales in notebook 05).

## 6. Annotating a Point
`ax.annotate()` points at one specific spot with text and an arrow. It's the best way to call out an outlier you found in Module 07:

In [ ]:
top = titanic.loc[fare.idxmax()]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(titanic['Age'], fare, alpha=0.4, color='tab:blue')
ax.annotate(
    f"£{top['Fare']:.0f}: the most expensive tickets\n(3 passengers shared one)",
    xy=(top['Age'], top['Fare']),            # the point being labeled
    xytext=(45, 420),                         # where the text goes
    arrowprops=dict(arrowstyle='->', color='black'),
)
ax.set(title='Age vs. Fare', xlabel='Age (years)', ylabel='Fare (£)')
plt.show()

`xy=` is the point being labeled, and `xytext=` is where the text sits, both in **data** coordinates. Pick an `xytext` in empty space.

## 7. Decluttering
Every line on a chart should earn its place. Two common cleanups: remove the top and right borders (**spines**), and use light gridlines on just the value axis:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, rates.values, color='tab:blue')
ax.bar_label(bars, labels=[f'{r:.0%}' for r in rates], padding=3)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set(title='Survival Rate by Class', ylim=(0, 1))

ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)          # draw the grid behind the bars
plt.show()

## 8. Multi-Panel Figures with a Loop
To compare groups side by side, give each group its own `ax`. `plt.subplots(1, 3)` returns an array of three axes, and `zip()` pairs each `ax` with its group:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True, layout='constrained')

for ax, (cls, group) in zip(axes, titanic.groupby('Pclass')):
    ages = group['Age'].dropna()
    ax.hist(ages, bins=range(0, 85, 5), color='tab:blue', edgecolor='white')
    ax.axvline(ages.median(), color='black', linestyle='--')
    ax.set(title=f'Class {cls}   (median {ages.median():.0f}, n = {len(ages)})', xlabel='Age (years)')

axes[0].set_ylabel('Passengers')
fig.suptitle('Age Distribution by Ticket Class')
fig.savefig(OUT / 'age_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

Three details that make panels comparable:
- **`sharey=True`**: all three panels use the same y-axis, so bar heights can be compared directly. Without it, each panel scales itself, and a small group looks as big as a large one.
- **The same bins** (`range(0, 85, 5)`) in every panel.
- **`layout='constrained'`**: Matplotlib spaces the panels so titles and labels don't overlap (a modern replacement for `fig.tight_layout()`).

## 9. Saving and Closing
`fig.savefig()` saves the whole figure. Call it **before** `plt.show()`, because in some environments showing a figure also clears it.

| Argument | Purpose |
|---|---|
| `dpi=150` | resolution (300 for print) |
| `bbox_inches='tight'` | trim extra white space |
| file extension | `.png` for images, `.svg`/`.pdf` for sharp vector graphics |

When you make many figures in a loop, close each one with `plt.close(fig)` so they don't pile up in memory.

In [ ]:
for cls, group in titanic.groupby('Pclass'):
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.hist(group['Fare'], bins=20, color='tab:blue', edgecolor='white')
    ax.set(title=f'Fares, Class {cls}', xlabel='Fare (£)', ylabel='Passengers')
    fig.savefig(OUT / f'fares_class_{cls}.png', dpi=100, bbox_inches='tight')
    plt.close(fig)     # saved, not shown

print(sorted(p.name for p in OUT.glob('fares_*.png')))

## 10. Functions That Draw on an `ax`
The OO style's biggest payoff: a function can take an `ax` and draw on it, and you can reuse it in any figure, in any panel. That's Module 03b's "write it once" idea applied to charts:

In [ ]:
def plot_distribution(ax, series, title, unit=''):
    """Histogram of a numeric Series on ax, with its mean and median marked."""
    values = series.dropna()
    ax.hist(values, bins=25, color='lightgray', edgecolor='white')
    ax.axvline(values.mean(), color='tab:red', linestyle='--', label=f'mean {values.mean():.1f}')
    ax.axvline(values.median(), color='tab:blue', label=f'median {values.median():.1f}')
    ax.set(title=title, xlabel=unit)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False)


fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
plot_distribution(axes[0], titanic['Age'], 'Age', 'years')
plot_distribution(axes[1], titanic['Fare'], 'Fare', '£')
plt.show()

---
### 📝 Summary
- `plt` creates and shows, `fig` holds the whole canvas (suptitle, layout, saving), and `ax` is one chart.
- `ax.set(title=..., xlabel=..., ylabel=..., ylim=...)` sets many properties at once.
- Put statistics on the chart with `axvline`/`axhline`/`axvspan`.
- Use `set_xscale('log')` (with `np.geomspace` bins) for skewed data, and format ticks with `StrMethodFormatter` and `PercentFormatter`.
- `ax.annotate()` calls out a point, `ax.bar_label()` labels bars, and removing spines and grid clutter helps.
- Panels: `plt.subplots(rows, cols, sharey=True, layout='constrained')` plus a `zip()` loop.
- `fig.savefig()` before `plt.show()`, and `plt.close(fig)` in loops.

## 🏋️ Practice

### Practice 1 — A Labeled Bar Chart (easy)
Load `pokemon.csv` into `poke`. Make a bar chart of the **number of Pokémon per Generation**, with each bar labeled by its count (`ax.bar_label`), a title, axis labels, and no top/right spines.

In [ ]:
poke = pd.read_csv('pokemon.csv')
# Your code here

### Practice 2 — Reuse the Function (medium)
Use `plot_distribution()` to build a **2 × 3** figure with one panel for each of the six Pokémon stats (`HP`, `Attack`, `Defense`, `Sp. Atk`, `Sp. Def`, `Speed`). Loop with `zip(axes.flat, stats)`. Add a `fig.suptitle()`, and save it as `OUT / 'pokemon_stats.png'`.

In [ ]:
# Your code here

### Practice 3 — Annotate the Outlier (harder)
Make a scatter plot of `Attack` vs. `Defense`. Find the Pokémon with the highest `Defense` (`.idxmax()`), and annotate it with its name and value. Add horizontal and vertical reference lines at the **median** Attack and Defense, splitting the chart into four quadrants. In a Markdown cell, describe what's in each quadrant.

In [ ]:
# Your code here

*Your description here.*

## ✏️ Your Turn
Make a figure that answers: **"How does survival rate differ by sex within each class?"** Use a 1 × 3 layout (one panel per class) with `sharey=True`, two bars per panel (female, male), percent-formatted y-axes, bar labels, and a `suptitle`. Save it as `OUT / 'survival_by_sex_and_class.png'`.

In [ ]:
# Your turn here

## 🔥 Challenge (Optional) — `subplot_mosaic` Layouts
`plt.subplot_mosaic()` builds uneven layouts from a picture made of letters:
```python
fig, axd = plt.subplot_mosaic([['big', 'big'],
                               ['left', 'right']], figsize=(10, 7), layout='constrained')
axd['big'].scatter(...)
```
Build a mosaic with a large Age-vs-Fare scatter on top and the Age and Fare histograms underneath. Save it as `OUT / 'titanic_mosaic.png'`.

In [ ]:
# Your code here

## 📦 Check Your Output Files
Run this last. Everything you saved should be inside `01-OutputFiles`. Upload that folder along with this notebook.

In [ ]:
for path in sorted(OUT.rglob('*')):
    print(path)

---
✅ Next up → [Plotting with Pandas](02-Plotting_with_Pandas.ipynb): one-line charts straight from a DataFrame, which are still Matplotlib `ax` objects underneath.